# 03 Features: one row per monotown

This notebook builds the variables used for clustering in notebook 04.
For each of the 247 matched monotowns we calculate:

- population 2025 and population change 2017-2025 (%)
- net migration per 1,000 residents (average of 2021-2024)
- average monthly wage 2025
- jobs per 1,000 residents 2025 and job change 2017-2025 (%)
- industry dependence: share of jobs in mining and manufacturing
- English names for towns, regions and municipality types

Input: the cleaned files from `data/processed/` (notebook 01) and the matched list `monotowns_oktmo.csv` (notebook 02).

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED = Path("../data/processed")
pd.set_option("display.max_columns", 50)

## Load the data and check the loaded files

OKTMO is the Russian code for municipalities (like a postcode for local governments).
We read it as text, not as a number, because some codes start with 0 (e.g. 04712000 for Yeniseysk),
and reading them as numbers would silently drop that zero and break the matching.

In [9]:
indicator_files = ["population", "migration", "wages", "employees", "industry", "profitable"]

data = {
    name: pd.read_csv(PROCESSED / f"{name}.csv", dtype={"oktmo": str, "oktmo_stable": str})
    for name in indicator_files
}

mono = pd.read_csv(PROCESSED / "monotowns_oktmo.csv", dtype={"oktmo": str})

In [12]:
for name, df in data.items():
    years = sorted(df["year"].unique())
    lengths = df["oktmo"].str.len().value_counts().to_dict()
    print(f"{name:11} rows={len(df):>7}  years={years}  oktmo lengths={lengths}")

print()
for name in ["population", "migration", "wages", "employees", "profitable"]:
    dups = data[name].duplicated(["oktmo", "year"]).sum()
    print(f"{name:11} duplicates (oktmo, year): {dups}")

population  rows=  30731  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 30731}
migration   rows=  58560  years=[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]  oktmo lengths={8: 58560}
wages       rows=   2486  years=[np.int64(2025)]  oktmo lengths={8: 2486}
employees   rows=   4927  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 4927}
industry    rows=  12021  years=[np.int64(2017), np.int64(2025)]  oktmo lengths={8: 12021}
profitable  rows=  13476  years=[np.int64(2024)]  oktmo lengths={8: 13476}

population  duplicates (oktmo, year): 0
migration   duplicates (oktmo, year): 0
wages       duplicates (oktmo, year): 0
employees   duplicates (oktmo, year): 0
profitable  duplicates (oktmo, year): 0


In [14]:
print(mono["note"].value_counts(), "\n")

matched = mono[mono["oktmo"].notna()].copy()
print("matched towns:", len(matched))                          # expected 247
print("duplicated oktmo:", matched["oktmo"].duplicated().sum())  # expected 0
print("oktmo lengths:", matched["oktmo"].str.len().value_counts().to_dict())

note
automatic                                       213
manual correction                                15
excluded: region not in Rosstat data              9
matched via migration table                       9
excluded: closed city (ZATO), no public data      7
district proxy                                    6
city part of merged okrug                         4
excluded: district proxy not representative       1
Name: count, dtype: int64 

matched towns: 247
duplicated oktmo: 0
oktmo lengths: {8: 247}


## Population and population change 2017-2025

Population is pivoted from one row per year to one row per town (`pop_2017`, `pop_2025`) and joined
to the 247 monotowns by OKTMO. Change is in percent: -15 means the town lost 15% of its residents.

Some towns are expected to lack values: those matched via the migration table (Crimea, Buryatia
and Dagestan are missing from the 2025 file) and municipalities whose OKTMO code changed between
2017 and 2025 (e.g. a district turned into an okrug). The dataset's `oktmo_stable` column links the
same territory across code changes, so we check whether it can fill these gaps.

In [24]:
pop = data["population"]

pop_wide = (
    pop.pivot(index="oktmo", columns="year", values="indicator_value")
       .rename(columns={2017: "pop_2017", 2025: "pop_2025"})
       .reset_index()
)
matched["category"] = pd.cut(matched["number"], bins=[0, 52, 201, 264], labels=[1, 2, 3]).astype(int)
features = matched[["number", "mono_name", "region", "category", "note", "ref_type", "oktmo"]].merge(
    pop_wide, on="oktmo", how="left"
)
features["pop_change_pct"] = (features["pop_2025"] / features["pop_2017"] - 1) * 100

print(features[["pop_2017", "pop_2025"]].isna().sum())
missing_pop = features["pop_2017"].isna() | features["pop_2025"].isna()
features.loc[missing_pop, ["mono_name", "region", "note", "ref_type", "oktmo", "pop_2017", "pop_2025"]]

pop_2017    71
pop_2025     7
dtype: int64


,mono_name,region,note,ref_type,oktmo,pop_2017,pop_2025
1,Онежский муниципальный округ Архангельской обл...,Архангельская область,automatic,Муниципальный округ,11546000,NaN,22829.0
4,Петровск-Забайкальский муниципальный округ Заб...,Забайкальский край,automatic,Муниципальный округ,76536000,NaN,28708.0
9,Междуреченский муниципальный округ Кемеровской...,Кемеровская область - Кузбасс,automatic,Муниципальный округ,32517000,NaN,95550.0
13,Топкинский муниципальный округ Кемеровской обл...,Кемеровская область - Кузбасс,automatic,Муниципальный округ,32531000,NaN,40199.0
16,Муниципальное образование Мантуровский муницип...,Костромская область,automatic,Муниципальный округ,34520000,NaN,14900.0
...,...,...,...,...,...,...,...
219,Атяшевское городское поселение Атяшевского мун...,Республика Мордовия,automatic,Городское поселение,89607151,NaN,6650.0
226,Касимовский муниципальный округ Рязанской области,Рязанская область,automatic,Муниципальный округ,61508000,NaN,53657.0
236,"Муниципальное образование ""Дорогобужский муниц...",Смоленская область,automatic,Муниципальный округ,66514000,NaN,23189.0
237,Городской округ город Невинномысск Ставропольс...,Ставропольский край,automatic,"Городской округ, городской округ с внутригород...",07724000,NaN,113112.0


In [32]:
codes = pop[["oktmo", "oktmo_stable"]].drop_duplicates()
check = features[["mono_name", "oktmo"]].merge(codes, on="oktmo", how="left")

print("towns with oktmo_stable different from oktmo:", (check["oktmo"] != check["oktmo_stable"]).sum())
check[check["oktmo"] != check["oktmo_stable"]]

towns with oktmo_stable different from oktmo: 106


,mono_name,oktmo,oktmo_stable
3,"Муниципальное образование ""Теплоозерское город...",99620170,CD
14,Муниципальное образование Демьяновское городск...,33632154,CD
17,Енисейский муниципальный округ Красноярского края,04712000,04515000
18,Парфинский муниципальный округ Новгородской об...,49630000,49530000
19,Городское поселение рабочий поселок Горный Тог...,50652153,CD
...,...,...,...
232,Малышевский муниципальный округ Свердловской о...,65762000,65535000
233,Муниципальный округ Ревда Свердловской области,65719000,65525000
234,Серовский муниципальный округ Свердловской обл...,65756000,65549000
235,Муниципальный округ Первоуральск Свердловской ...,65753000,65524000


In [40]:
def find(root):
    rows = pop["municipality"].str.contains(root, case=False, na=False)
    return pop.loc[rows, ["year", "municipality", "mun_type", "oktmo", "oktmo_stable", "indicator_value"]].sort_values(["year", "oktmo"])

for root in ["Алатыр", "Мантуров", "Краснокаменск", "Невинномысск", "Рузаев"]:
    print(f"\n=== {root} ===")
    print(find(root).to_string(index=False))


=== Алатыр ===
 year                   municipality                                                    mun_type    oktmo oktmo_stable  indicator_value
 2017 Алатырский муниципальный район                                         Муниципальный район 97603000     97503000          14875.0
 2017                  Город Алатырь Городской округ, городской округ с внутригородским делением 97704000     97503000          35298.0
 2025 Алатырский муниципальный округ                                         Муниципальный округ 97503000     97503000          12981.0
 2025                  Город Алатырь Городской округ, городской округ с внутригородским делением 97704000     97503000          30989.0

=== Мантуров ===
 year                     municipality                                                    mun_type    oktmo oktmo_stable  indicator_value
 2017 Мантуровский муниципальный район                                         Муниципальный район 34620000     34520000           4017.0
 2017     

## Population and population change 2017-2025

Population is joined to the monotowns by OKTMO and change is calculated in percent
(-15 = the town lost 15% of its residents). Many municipalities were reorganised between 2017 and 2025,
so gaps are filled in three steps (column `pop_source` records which one was used):

1. **Chuvashia fix:** Alatyr, Kanash and Shumerlya are matched to the town itself, because on
   1 January 2025 the municipal okrug code covered only the rural district (same approach as
   "city part of merged okrug" in notebook 02).
2. **Changed codes:** if a code has no 2017 value, 2017 population is looked up through `oktmo_stable`,
   which links old and new codes of the same territory. Merged units (e.g. town + district) are summed.
3. **District proxy:** Mordovian settlements have no 2017 data, so their district's change is used instead.

Remaining gaps (Zabaikalsky krai and Nevinnomyssk have no 2017 data; Crimea, Dagestan, Buryatia and
Tolyatti have no population data) are handled after all variables are built.
Note: 2017 estimates are based on the 2010 census and 2025 estimates on the 2021 census, so part of the
change reflects census corrections, not only real population loss.

In [43]:
#step 1: match the three Chuvash monotowns to the town instead of the okrug
for root, town in [("Алатыр", "Город Алатырь"), ("Канаш", "Город Канаш"), ("Шумерл", "Город Шумерля")]:
    town_code = pop.loc[(pop["year"] == 2025) & (pop["municipality"] == town), "oktmo"].iloc[0]
    rows = matched["mono_name"].str.contains(root)
    matched.loc[rows, ["oktmo", "note"]] = [town_code, "city part of merged okrug"]

print("duplicated oktmo:", matched["oktmo"].duplicated().sum())
matched.loc[matched["region"].str.contains("Чуваш"), ["mono_name", "oktmo", "note"]]

duplicated oktmo: 0


,mono_name,oktmo,note
194,Алатырский муниципальный округ Чувашской Респу...,97704000,city part of merged okrug
195,Канашский муниципальный округ Чувашской Респуб...,97707000,city part of merged okrug
196,Шумерлинский муниципальный округ Чувашской Рес...,97713000,city part of merged okrug


In [49]:
pop_wide = (
    pop.pivot(index="oktmo", columns="year", values="indicator_value")
       .rename(columns={2017: "pop_2017", 2025: "pop_2025"})
)

features = matched[["number", "mono_name", "region", "category", "note", "ref_type", "oktmo"]].merge(
    pop_wide, left_on="oktmo", right_index=True, how="left"
)
features["pop_source"] = np.where(features["pop_2017"].notna() & features["pop_2025"].notna(), "direct", None)

#step 2: missing 2017 values through oktmo_stable (summing merged units)
stable_2025 = pop.loc[pop["year"] == 2025].set_index("oktmo")["oktmo_stable"]
pop17_stable = (
    pop.loc[(pop["year"] == 2017) & (pop["oktmo_stable"] != "CD")]
       .groupby("oktmo_stable")["indicator_value"].sum()
)
gap = features["pop_2017"].isna()
features.loc[gap, "pop_2017"] = features.loc[gap, "oktmo"].map(stable_2025).map(pop17_stable)
features.loc[gap & features["pop_2017"].notna(), "pop_source"] = "2017 via oktmo_stable"

features["pop_change_pct"] = (features["pop_2025"] / features["pop_2017"] - 1) * 100

#step 3: settlements without 2017 data get their district's change
#district 2025 = sum of its settlements (same first 5 digits of OKTMO)
p25 = pop.loc[(pop["year"] == 2025) & ~pop["oktmo"].str.endswith("000")]
district_2025 = p25.groupby(p25["oktmo"].str[:5] + "000")["indicator_value"].sum()
district_change = (district_2025 / pop_wide["pop_2017"] - 1) * 100

gap = features["pop_change_pct"].isna() & features["pop_2025"].notna()
features.loc[gap, "pop_change_pct"] = (features.loc[gap, "oktmo"].str[:5] + "000").map(district_change)
features.loc[gap & features["pop_change_pct"].notna(), "pop_source"] = "district change (proxy)"

In [51]:
print(features["pop_source"].value_counts(dropna=False), "\n")
print(features["pop_change_pct"].describe().round(1))
features.loc[features["pop_change_pct"].isna(), ["mono_name", "region", "note", "pop_2017", "pop_2025"]]
cols = ["mono_name", "pop_source", "pop_2017", "pop_2025", "pop_change_pct"]
ranked = features.dropna(subset=["pop_change_pct"]).sort_values("pop_change_pct")
display(pd.concat([ranked.head(5), ranked.tail(5)])[cols])
features.loc[features["pop_source"] == "district change (proxy)", cols]

pop_source
direct                     174
2017 via oktmo_stable       54
NaN                         13
district change (proxy)      6
Name: count, dtype: int64 

count    234.0
mean      -9.1
std        7.9
min      -31.1
25%      -13.0
50%       -9.0
75%       -5.5
max       37.3
Name: pop_change_pct, dtype: float64


,mono_name,pop_source,pop_2017,pop_2025,pop_change_pct
212,Муниципальное образование Верхнекамский муници...,2017 via oktmo_stable,27783.0,19134.0,-31.130547
35,"Муниципальный округ ""Инта"" Республики Коми",2017 via oktmo_stable,28977.0,20198.0,-30.296442
20,Муниципальное образование городской округ горо...,direct,16922.0,12358.0,-26.970807
1,Онежский муниципальный округ Архангельской обл...,2017 via oktmo_stable,30762.0,22829.0,-25.788310
161,Городской округ город Абаза Республики Хакасия,direct,15592.0,11823.0,-24.172653
172,Городской округ Верхняя Пышма Свердловской обл...,direct,83017.0,92113.0,10.956792
193,Чебаркульский городской округ Челябинской области,direct,39914.0,44341.0,11.091346
209,"Муниципальное образование ""Янтарный городской ...",direct,6438.0,7198.0,11.804908
153,"Городское поселение ""Город Удачный"" муниципаль...",direct,11835.0,13957.0,17.929869
90,Боровский муниципальный округ Калужской области,direct,61620.0,84597.0,37.288218


,mono_name,pop_source,pop_2017,pop_2025,pop_change_pct
147,Городское поселение Рузаевка Рузаевского муниц...,district change (proxy),NaN,41660.0,-8.201250
148,Кадошкинское городское поселение Кадошкинского...,district change (proxy),NaN,3796.0,-3.959226
149,Комсомольское городское поселение Чамзинского ...,district change (proxy),NaN,11387.0,-6.581480
150,Торбеевское городское поселение Торбеевского м...,district change (proxy),NaN,9310.0,-3.715783
151,Тургеневское городское поселение Ардатовского ...,district change (proxy),NaN,4184.0,-7.506351
235,Атяшевское городское поселение Атяшевского мун...,district change (proxy),NaN,6650.0,-1.905869
